In [ ]:
import segmentation_models_pytorch as smp
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from torch.utils.data import DataLoader
from albumentations.pytorch import ToTensorV2
from torchvision.models.segmentation.deeplabv3 import DeepLabHead
from torchvision import models
import albumentations as A
from albumentations.pytorch import ToTensorV2
from albumentations import (RandomCrop, CenterCrop, ElasticTransform, RGBShift, Rotate,
                            Compose, ToFloat, FromFloat, RandomRotate90, Flip, OneOf, MotionBlur, MedianBlur, Blur,
                            Transpose,
                            ShiftScaleRotate, OpticalDistortion, GridDistortion, RandomBrightnessContrast, VerticalFlip,
                            HorizontalFlip,
                        HueSaturationValue
                        )
import shutil
import torch.nn as nn
import yaml
from transformers import (
    SegformerForSemanticSegmentation,
    TrainingArguments, Trainer,
    SegformerImageProcessor)
sys.path.append('..')
from dataset.segmentation import *
from config import *

AVAIL_GPUS = min(1, torch.cuda.device_count())
device = "cuda" if torch.cuda.is_available() else "cpu"
#device = 'cpu'

In [ ]:
num_classes = None
if num_classes == None:
    num_classes = len(os.listdir((os.path.join(data_path, "masks_classes"))))

In [ ]:
encoder = 'nvidia/mit-b5'

In [ ]:
processor = SegformerImageProcessor.from_pretrained(encoder)
with open('../preprocessing/class_mapping.yaml', 'r') as stream:
    try:
        # Converts yaml document to python object
        label2id = yaml.safe_load(stream)
    except yaml.YAMLError as e:
        print(e)
label2id['bkg'] = 0
id2label = {v: k for k, v in label2id.items()}

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(encoder,num_labels=num_classes + 1,
                                                        id2label=id2label,label2id=label2id,
                                                             ignore_mismatched_sizes=True)

In [ ]:
checkpoint_path = "../model_results/segformer_k_fold_multiclass/nvidia/mit-b5/fold_4/segformer_processor_decoder_w_1_3_2_2024_03_27_14_31_29/"

In [ ]:
checkpoint = torch.load(os.path.join(checkpoint_path, "model.pth"), map_location=torch.device(device))
model.load_state_dict(checkpoint['model_state_dict'], strict=False)
model.to(device)

In [ ]:
train=False
val=True
test=False

images_path = "../data/cropped_data/images"
data_path = Path(images_path).parent.as_posix()
normalize_imagenet = 0

if 'cfg' in checkpoint.keys():
    cfg = checkpoint['cfg']
    normalize_imagenet = cfg.dataset.normalize_imagenet
    print('from cfg normalize imagenet', normalize_imagenet)
else:
    print('not cfg', normalize_imagenet)

if not cfg.opt.processor:
    processor = None
else:
    processor = SegformerImageProcessor.from_pretrained(encoder)

shuffle=1
random_seed=123
num_workers=0
batch_size = 1

train_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "train")
val_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "val")
test_df_path = os.path.join(k_fold_data_path, f'fold_{cfg.dataset.fold}', "test")

transform = None


if train:
    dataset = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=train_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=processor)
    dataset_img = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=train_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=None)
elif val:
    dataset = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=val_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=processor)
    dataset_img = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=val_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=None)
elif test:
    dataset = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=test_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=processor)
    dataset_img = KFoldDataframeMulticlassProcessor_v2(data_path=data_path, df_path=test_df_path,
                                   transform=transform, cropped=cfg.dataset.cropped
                                                  ,processor=None)
    
dataloader_img = DataLoader(dataset_img, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                            drop_last=True)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers,
                            drop_last=True)

In [ ]:
results_path = os.path.join(checkpoint_path, "results")
if not os.path.exists(results_path):
    os.makedirs(results_path)
else:
    shutil.rmtree(results_path)
    os.makedirs(results_path)

In [ ]:
palette = [0, 150, 255]

In [ ]:
model.eval()

In [ ]:
save=False
threshold_value = 0.3

count = 0
model.eval()

with torch.no_grad():
    for i, zipped_1 in enumerate(zip(dataloader, dataloader_img)):
        image, mask = zipped_1[0]
        im_viz, _ = zipped_1[1]
        im = image.to(device)
        mask_batch = mask.to(device)
        
        logits = model(im.to(device)).logits

        upsampled_logits  = nn.functional.interpolate(
            logits,
            size=tuple(im.shape[-2:]),  # (height, width)
            mode='bilinear',
            align_corners=False
        )
        
        
        for ix in range(upsampled_logits.shape[0]):
            # Create subplots with one row and three columns
            mask = mask_batch[ix]
            pred = upsampled_logits[ix].softmax(0)
            
            pred = pred.permute(1,2,0).detach().cpu().numpy()
            #print(np.unique(pred))
            #pred = np.where(pred > threshold_value, pred, 0)

            pred = np.argmax(pred, 2)

            pred[pred == 0] = palette[0]
            pred[pred == 1] = palette[1]
            pred[pred == 2] = palette[2]

            mask[mask == 0] = palette[0]
            mask[mask == 1] = palette[1]
            mask[mask == 2] = palette[2]

            print(np.unique(mask.detach().cpu().numpy()))
            print(np.unique(pred))

            if 150 in mask or 150 in pred:
            
                fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
                # Plot on the first subplot
                axes[0].imshow(im_viz[ix].permute(1,2,0).detach().cpu().numpy())
                axes[0].set_title('Plot 1')
    
                # Plot on the second subplot
                axes[1].imshow(mask.detach().cpu().numpy())
                axes[1].set_title('Plot 2')
    
                # Plot on the third subplot
                axes[2].imshow(pred)
                axes[2].set_title('Plot 3')
    
                # Adjust layout to prevent clipping of titles
                plt.tight_layout()
    
                # Show the plots
                plt.show()
                
    
                name = dataset.images_file_names[count]
                if save:
                    #print(name)
                    cv2.imwrite(os.path.join(results_path, f"{name}"), np.squeeze(thresholded_image*255))
                    #plt.imsave(os.path.join(results_path, f"{name}"),np.squeeze(thresholded_image), cmap='gray')
                    
                count += 1

In [ ]:
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
loss = 0
for i, (image, mask) in enumerate(dataloader):

    im = image.to(device)
    mask_batch = mask.to(device)
    
    # logits = model(im.to(device)).logits

    outputs = model(pixel_values=im, labels=mask_batch.long()).logits
    # loss, logits = outputs.loss, outputs.logits

    # outputs = model(inputs.squeeze()).logits
    upsampled_logits = nn.functional.interpolate(
        outputs,
        size=tuple(im.shape[-2:]),  # (height, width)
        mode='bilinear',
        align_corners=False
    )
    
    loss += criterion(upsampled_logits.float(), mask_batch.squeeze(1).long()).item()
    print(loss/(i+1))
    if i > 1000:
        break

In [ ]:
results_path = os.path.join(checkpoint, "heatmap_results")
if not os.path.exists(results_path):
    os.makedirs(results_path)
else:
    shutil.rmtree(results_path)
    os.makedirs(results_path)

In [ ]:
import gc

count = 0
model.eval()


with torch.no_grad():
    for i, (im, mask) in enumerate(dataloader):
        print(i)
        results = model(im.to(device))
        
        
        for ix in range(results['out'].shape[0]):
            # Create subplots with one row and three columns

            pred = results['out'][ix].sigmoid().permute(1,2,0).detach().cpu().numpy()
            name = dataset.images_file_names[count]

            plt.imsave(os.path.join(results_path, f"{name}"),np.squeeze(pred), cmap='gray')
            count += 1